# How we know the numbers are right

Every other notebook here produces a number. This one is about whether to believe
them.

That is not a rhetorical framing. Auditing this project produced 21 findings, and
**every one is the same shape: code that ran without error, returned a plausible
value, and was wrong.** Not a crash, not a warning, not a NaN. A number that looked
exactly like a number.

A crash is cheap. Someone notices, someone fixes it. A confident wrong number is
expensive, because it propagates into a table, into a claim, into a submission,
and the first person to catch it may be a reviewer.

This notebook shows the failure mode concretely, then shows the checks that now
stand between it and a published table.

In [1]:
import os
import sys
from pathlib import Path

# Anchor at the repository root so every default path in psbd resolves the same
# way it does from a script, whichever directory the notebook was opened from.
REPO_ROOT = next(
    parent for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "pyproject.toml").exists()
)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import torch

## Failure 1. A detector pointing backwards

Every detector here returns a score where **low means poisoned**, which is what
`defences.decision.detection_report` assumes. Some published statistics already point
that way and some point the opposite way, so each port negates exactly once, at
one place, and says so in its docstring.

The trap is that the family is not uniform. STRIP's raw entropy already points the
right way. SCALE-UP's, IBD-PSC's and TeCo's do not. Negate everything and 3 of 4
are wrong; negate nothing and 1 of 4 is wrong.

That is not hypothetical. Wiring these up the first time inverted 2 of 3.

In [2]:
from detectors import DETECTOR_NAMES, FORWARD_PASSES_PER_INPUT

registry = pd.DataFrame(
    {
        "detector": list(DETECTOR_NAMES),
        "forward_passes_per_input": [FORWARD_PASSES_PER_INPUT[n] for n in DETECTOR_NAMES],
    }
)
registry

,detector,forward_passes_per_input
0,confidence,1
1,strip,8
2,scale_up,6
3,scale_up_data_limited,6
4,ibd_psc,6
5,teco,71


### What the bug looked like

These are the AUROC values the first wiring produced, against what the same code
produces once the sign is right. Nothing raised in either case.

In [3]:
observed = pd.DataFrame(
    [
        {"detector": "confidence", "as_wired": 0.878, "corrected": 0.878, "published": None},
        {"detector": "strip", "as_wired": 1.000, "corrected": 1.000, "published": None},
        {"detector": "scale_up", "as_wired": 0.689, "corrected": 0.689, "published": 0.93},
        {"detector": "ibd_psc", "as_wired": 0.043, "corrected": 0.957, "published": 0.99},
        {"detector": "teco", "as_wired": 0.055, "corrected": 0.945, "published": 0.92},
    ]
).set_index("detector")
observed["1 - as_wired"] = 1 - observed["as_wired"]
observed

,as_wired,corrected,published,1 - as_wired
detector,,,,
confidence,0.878,0.878,NaN,0.122
strip,1.000,1.000,NaN,0.000
scale_up,0.689,0.689,0.93,0.311
ibd_psc,0.043,0.957,0.99,0.957
teco,0.055,0.945,0.92,0.945


Read the last column. An inverted detector does not return noise, it returns
**one minus the right answer**. 0.043 and 0.957 are the same measurement with a
sign error, and only one of them looks wrong to a human reading a table.

The `published` column is what made the diagnosis certain rather than a guess:
flipping those two lands on the values their papers report.

## The check that catches it

A published detector, run on a model with a working backdoor, must score above
chance. That is mechanical, needs no ground truth beyond "this attack works", and
would have caught both inversions immediately.

It runs against a backdoor we install ourselves, so a failure means the **code** is
wrong rather than the checkpoint being unusual. CPU, seconds.

In [4]:
from experiments.preflight.synthetic import (
    attack_success_rate,
    build_backdoored_model,
    build_splits,
)

device = torch.device("cpu")
model = build_backdoored_model()
loaders = build_splits(num_samples=192)

print(f"synthetic backdoor ASR: {attack_success_rate(model, loaders, device):.3f}")
print("every triggered input reaches the target class, so any detector that is")
print("wired correctly must find it")

synthetic backdoor ASR: 1.000
every triggered input reaches the target class, so any detector that is
wired correctly must find it


In [5]:
from defences.decision import HEADLINE_QUANTILE, detection_report
from detectors import DetectorContext, build_detector

CHEAP_CORRUPTIONS = ("gaussian_noise", "defocus_blur", "brightness", "contrast")
context = DetectorContext(
    model=model,
    device=device,
    mean=(0.0, 0.0, 0.0),
    std=(1.0, 1.0, 1.0),
    validation_loader=loaders["validation"],
    num_classes=10,
    use_bfloat16=False,
    teco_corruptions=CHEAP_CORRUPTIONS,
)

rows = []
for name in ("confidence", "strip", "ibd_psc", "teco"):
    score = build_detector(name, context)
    scores = {s: score(model, loader, device) for s, loader in loaders.items()}
    upright = detection_report(
        scores["validation"], scores["clean"], scores["backdoor"], HEADLINE_QUANTILE
    )["auroc"]
    inverted = detection_report(
        -scores["validation"], -scores["clean"], -scores["backdoor"], HEADLINE_QUANTILE
    )["auroc"]
    rows.append({"detector": name, "as_written": upright, "if_negated": inverted})

pd.DataFrame(rows).set_index("detector").round(4)

Seed set to 0


Seed set to 0


Seed set to 0


Seed set to 0


Seed set to 0


Seed set to 0


Seed set to 0


,as_written,if_negated
detector,,
confidence,1.0000,0.0000
strip,0.6619,0.3381
ibd_psc,1.0000,0.0000
teco,0.9162,0.0838


The right-hand column is the bug, reproduced on demand. `tests/test_preflight.py`
asserts that column stays below 0.40, so the check is itself checked. A gate nobody
has seen fail is a gate nobody knows works.

### What the gate refuses to judge

It also has to know when it *cannot* answer. SCALE-UP's statistic takes 6 values
over 5 amplification scales, and this synthetic model keeps almost every clean
prediction stable, so nearly every pair is tied and the AUROC sits at chance for
reasons that have nothing to do with its sign.

Calling that "inverted" would be a false alarm. The gate measures how concentrated
the clean scores are and reports **not exercised** instead, which sends the
question to a real checkpoint rather than answering it wrongly.

In [6]:
score = build_detector("scale_up", context)
clean = score(model, loaders["clean"], device)
values, counts = clean.unique(return_counts=True)

print(f"distinct SCALE-UP scores across {len(clean)} clean samples: {len(values)}")
print(f"fraction sitting on the single most common value: {counts.max() / clean.numel():.1%}")
print()
print("with that many ties the ranking cannot separate anything, so the sign")
print("cannot be read off this case at all")

distinct SCALE-UP scores across 192 clean samples: 2
fraction sitting on the single most common value: 96.9%

with that many ties the ranking cannot separate anything, so the sign
cannot be read off this case at all


## Failure 2. Records that outlived their data

The sweep writes tensors, then a second stage reads them and writes a summary. If
the tensors move, the summary entries stay behind and nothing notices, because a
record computed from deleted data looks exactly like one computed from present
data.

That happened here. A superseded operator's caches were archived, and **3029
summary rows produced from them stayed in the table**, labelled as ordinary
measurements. They score higher than the records that survive, so they inflated
whatever they were averaged into.

The fix is a column recording whether the source tensors still exist, and a loader
that drops those rows unless asked not to.

In [7]:
from evaluation.summary import load_detection_summary, summary_coverage

safe = load_detection_summary()
everything = load_detection_summary(
    plain_only=False, require_cache_backed=False, verbose=False
)

comparison = pd.DataFrame(
    [
        {
            "rows": len(everything),
            "dropout mean auroc": everything.loc[everything.operator == "dropout", "auroc"].mean(),
            "overall mean auroc": everything["auroc"].mean(),
            "table": "unfiltered",
        },
        {
            "rows": len(safe),
            "dropout mean auroc": safe.loc[safe.operator == "dropout", "auroc"].mean(),
            "overall mean auroc": safe["auroc"].mean(),
            "table": "plain and cache-backed only",
        },
    ]
).set_index("table")
comparison.round(4)

detection summary: 24043 of 30378 rows kept, dropped 6335 variant, 0 not cache-backed


,rows,dropout mean auroc,overall mean auroc
table,,,
unfiltered,30378,0.7403,0.7383
plain and cache-backed only,24043,0.7222,0.7240


`dropout` is the published baseline every reported margin is measured against, so
contaminating it does not make our numbers look worse. **It makes them look
better**, by inflating the thing they are compared to. That is the direction of
error least likely to be noticed by the people it flatters.

## Coverage, which is worth looking at before any average

An average over an unequal grid is not the number it appears to be. This project
has had conclusions inverted by exactly that.

In [8]:
summary_coverage(safe)

,architecture,dataset,checkpoints,cells
0,swin,cifar10,25,737
1,swin,cifar100,57,1741
2,swin,tiny,30,720
3,vit,cifar10,40,6067
4,vit,cifar100,57,5895
5,vit,gtsrb,22,3984
6,vit,tiny,52,4899


## Failure 3. Comparing at unmatched strength

The largest single correction was a headline that compared two placements read at
completely different perturbation strengths.

The rule this project set for itself is that two placements may only be compared
at a matched clean-validation shift ratio, because the nominal rate means
different things at different sites. The headline broke that rule, and the size of
the mismatch was the size of the claimed effect.

In [9]:
withdrawn = pd.DataFrame(
    [
        {"arm": "dropout @ pre_residual (baseline)", "rate": 0.3, "shift ratio": 0.709},
        {"arm": "gain_scale @ mlp_norm_out (winner)", "rate": 2.0, "shift ratio": 0.964},
        {"arm": "token_mask @ before_attention_norm", "rate": 0.4, "shift ratio": 0.713},
    ]
).set_index("arm")
print("The withdrawn comparison, and the one that replaced it:\n")
print(withdrawn.to_string())
print()
print("claimed gain for gain_scale, unmatched      +0.258")
print("its gain at matched shift ratio, full panel -0.007")
print()
print("token_mask's own mismatch against the baseline is -0.048, meaning it is")
print("read at the LOWER disturbance, so its +0.089 is conservative rather than")
print("flattered")

The withdrawn comparison, and the one that replaced it:

                                    rate  shift ratio
arm                                                  
dropout @ pre_residual (baseline)    0.3        0.709
gain_scale @ mlp_norm_out (winner)   2.0        0.964
token_mask @ before_attention_norm   0.4        0.713

claimed gain for gain_scale, unmatched      +0.258
its gain at matched shift ratio, full panel -0.007

token_mask's own mismatch against the baseline is -0.048, meaning it is
read at the LOWER disturbance, so its +0.089 is conservative rather than
flattered


## Failure 4. A run that succeeds and writes nothing readable

The first 3 failures all produced a wrong number. This one produced no number at
all, and that turned out to be harder to notice rather than easier.

Every job generator is supposed to pass `--output checkpoints/<name>/attack_result.pt`.
Two of them passed `--output checkpoints/<name>`. `save_checkpoint` used
`os.path.dirname(path)` to place the sidecar, so with that spelling:

- the weights landed in a 343 MB plain **file** named `<name>`, where nothing looks
  for them, and
- `args.json` landed at `checkpoints/args.json`, which every later affected run
  overwrote.

The training itself was fine. The job exited 0, printed its final ASR to the log,
and took several GPU-hours doing real work. What it left behind was unreadable by
every downstream tool.

**Nothing reported this for 4 weeks.** `coverage_ledger.py` walks
`checkpoints/` with `os.listdir` and descends into directories, so a plain file is
not a broken cell, it is not a cell. The ledger's job is to report cells that are
missing, and these were missing in exactly the way an unrun experiment is missing.

21 runs were affected. 7 of them belonged to the WaNet strength sweep that was
running at the time, with 14 more jobs queued behind them, so the sweep was
destroying its own results as it produced them.

In [10]:
from train import resolve_checkpoint_path

# The fix went into save_checkpoint rather than into the 2 generators, because PBS
# copies a job script at submission time: editing the queued .pbs files would not
# have reached the 14 jobs already in the queue, but changing the function they
# call at runtime does.
for spelling in ("checkpoints/vit_gtsrb_wanet_0_01_trig_s4",
                 "checkpoints/vit_gtsrb_wanet_0_01_trig_s4/attack_result.pt"):
    print(f"{spelling:<58} -> {resolve_checkpoint_path(spelling)}")

# What the recovery got back. Provenance came from 2 sources the bug never touched:
# the job script that launched each run (every flag, including --attack-override)
# and its log line (final ASR=... CA=...).
recovered = pd.DataFrame(
    [
        {"batch": "WaNet and BadNet strength sweep", "runs": 8, "config": "yes", "asr": "yes"},
        {"batch": "_v2 retrain pilot", "runs": 2, "config": "yes", "asr": "no"},
        {"batch": "August evade and dropout experiments", "runs": 11, "config": "6 of 11", "asr": "no"},
    ]
).set_index("batch")
print()
print(recovered.to_string())
print()
print("the 8 fully recovered runs are the WaNet dose-response, and one of them")
print("is the first WaNet cell to clear ASR 0.85 at 1 percent poisoning")

checkpoints/vit_gtsrb_wanet_0_01_trig_s4                   -> checkpoints/vit_gtsrb_wanet_0_01_trig_s4/attack_result.pt
checkpoints/vit_gtsrb_wanet_0_01_trig_s4/attack_result.pt  -> checkpoints/vit_gtsrb_wanet_0_01_trig_s4/attack_result.pt

                                      runs   config  asr
batch                                                   
WaNet and BadNet strength sweep          8      yes  yes
_v2 retrain pilot                        2      yes   no
August evade and dropout experiments    11  6 of 11   no

the 8 fully recovered runs are the WaNet dose-response, and one of them
is the first WaNet cell to clear ASR 0.85 at 1 percent poisoning


## Failure 5. A flag that kept only the last value

This one was caught while verifying the fix for a different problem, which is the
only reason it is here rather than in a table.

Label-Consistent needs 2 configuration values: where its adversarially perturbed
base images live, and the perturbation strength those bases carry. The natural way
to write that is 2 flags:

```
--attack-override adversarial_dir=results/lc_adversarial/cifar100_tl0_eps16 \
--attack-override adversarial_epsilon=0.062745
```

`--attack-override` was declared `nargs="*"`, which is not an accumulating action.
Argparse therefore **replaced** the first occurrence with the second, and only
`adversarial_epsilon` survived. With `adversarial_dir` empty the attack falls back
to the patch-only variant, so the run would have trained the weak attack while its
`args.json` advertised the strong one.

The end-to-end smoke run passed. Training completed, ASR and clean accuracy were
recorded, and the coverage check for missing bases did not fire, because with no
directory configured there is nothing to be missing. The only visible symptom was
one absent key in a metadata dict.

Three changes, because the flag was only the surface:

1. The flag now uses `action="extend"`, so both spellings accumulate.
2. The job generator passes both keys in a single flag anyway.
3. **A configuration carrying an epsilon but no directory is now refused before
   training starts.** That is the invariant the other 2 fixes cannot supply: it
   catches the incoherent state whatever produced it.

In [11]:
import argparse

# The bug and its fix, side by side. Nothing about the calling convention changed,
# only whether argparse accumulates.
repeated = ["--attack-override", "adversarial_dir=bases/", "--attack-override", "adversarial_epsilon=0.0627"]

buggy = argparse.ArgumentParser()
buggy.add_argument("--attack-override", nargs="*", default=None)
fixed = argparse.ArgumentParser()
fixed.add_argument("--attack-override", action="extend", nargs="*", default=[])

print("as written on the command line:")
print(f"  {' '.join(repeated)}\n")
print(f"  nargs='*'            -> {buggy.parse_args(repeated).attack_override}")
print(f"  action='extend'      -> {fixed.parse_args(repeated).attack_override}")
print()
print("the dropped key is adversarial_dir, and dropping it turns the attack back")
print("into the patch-only variant without any error")

as written on the command line:
  --attack-override adversarial_dir=bases/ --attack-override adversarial_epsilon=0.0627

  nargs='*'            -> ['adversarial_epsilon=0.0627']
  action='extend'      -> ['adversarial_dir=bases/', 'adversarial_epsilon=0.0627']

the dropped key is adversarial_dir, and dropping it turns the attack back
into the patch-only variant without any error


## The pattern

Five failures, one shape.

| failure | what it returned | why nothing caught it |
|---|---|---|
| inverted detector | 1 minus the right answer | no test asserted a direction |
| stale records | a real number from deleted data | the column marking them had no reader |
| unmatched comparison | a real gain, of the wrong thing | the achieved strength was never printed beside the result |
| unreadable checkpoint | nothing, from hours of real training | a malformed cell is indistinguishable from an unrun one |
| dropped config flag | a real ASR, for the weaker attack | the weak variant is a legitimate configuration, so nothing was out of range |

None of these is a coding error in the ordinary sense. Each is a place where the
code could not tell the difference between a right answer and a wrong one, so it
returned whichever it got. The fourth extends that: the code could not tell the
difference between work it had destroyed and work it had never been asked to do.

The fifth is the sharpest version, and it is worth sitting with. The dropped flag
left the attack in a state that is **valid**. Patch-only Label-Consistent is a real
configuration that this project trained deliberately for months. Nothing was out of
range, nothing was null, no invariant was violated. The run was wrong only relative
to an intent recorded nowhere the code could read it.

The checks that now exist are not more care. Care is what failed, 5 times. They are
mechanical questions with mechanical answers: does this detector beat chance on a
backdoor we installed, do these records still have their tensors, were these two
arms read at the same strength, does this output path name a folder, does this
config carry a strength without the data that strength describes. Each one fails
loudly instead of returning a number, and each is tested by reintroducing the bug
it exists to catch.

The general rule these converge on: **make the incoherent state unrepresentable,
or make it loud.** A pipeline this size will keep producing plausible wrong
numbers, and the only durable defence is a machine-checkable statement of what the
run was supposed to be, checked before it starts rather than after it finishes.